# S5.M10-1 主动方明文直方图在两进程生产模式下的墙钟

**目的**：方法筛选第五阶段第 2 步（[DR-M5-016](../../../registry/decision_records/DR-M5-016.yaml) · D-7）。

S5.M7 测的是 debug_mode 单进程模拟：两方的计算串行相加，得到的是「两方合计计算时间」。真实部署中两方并行，被动方可能在关键路径上，墙钟节省可能更小。

本步在会话容器内起两个本地进程，各自一方：
- 用 `sf.init(cluster_config, ray_mode=False)` 的生产模式，本机回环通信；
- 每个进程只读自己那一方的特征，标签只在内地进程；
- 明文直方图替换在两个进程里同样开启，被动方收到的消息不变。

每个任务的测法：
- 先做一次 2 棵树的预热（不记录）；
- 再按原版、明文、明文、原版各训前 10 棵树（Orange 大版 5 棵）；
- 记录两方训练墙钟、启动到结束的总墙钟与回环网卡发送字节数；四次测试打分须两两逐位相同。

**口径**：容器 2 核、本机回环通信，不代表跨境网络或汇丰环境；只作两种实现的相对比较。

**数据**：公开数据，属开发证据。

In [1]:
import hashlib
import json
import logging
import os
import pathlib
import subprocess
import sys

import numpy as np
import pandas as pd
import yaml

HASH_PREFIX_LEN = 12
logging.getLogger().setLevel(logging.ERROR)        # 隐语运行日志不进 notebook
REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / "AGENTS.md").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
CONFIG_PATH = REPO_ROOT / "modules/m5_modeling/configs/s5_m10_wallclock.yaml"
m10 = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
cfg = yaml.safe_load((REPO_ROOT / m10["base_config"]).read_text(encoding="utf-8"))
COMPONENTS = [REPO_ROOT / "modules/m5_modeling/components" / n
              for n in ("sgb_wallclock.py", "sgb_cost_screen.py", "method_screen_sgb.py", "method_screen.py", "g3_ssxgb.py")]


def sha(path, n=HASH_PREFIX_LEN):
    return hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest()[:n]


git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True, cwd=REPO_ROOT).strip()
dirty = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True,
                       cwd=REPO_ROOT).stdout.strip() != ""
for p in [CONFIG_PATH, REPO_ROOT / m10["decision_record"]] + COMPONENTS:
    print(p.relative_to(REPO_ROOT), "sha256:" + sha(p))
print("决策记录:", m10["decision_record"], "| step:", m10["step_id"], "|", m10["purpose"])
print("seed:", m10["seed"], "| git:", git_sha + ("（工作区含本步未提交改动）" if dirty else ""))
PRIV = pathlib.Path(os.environ[cfg["private_root_env"]])
RUN = pathlib.Path(os.environ[cfg["run_dir_env"]])
LARGE, M10 = (pathlib.Path(os.environ[m10[k]]) for k in ("large_dir_env", "out_dir_env"))
launch, n_seg = {}, 0
for line in (M10 / "fingerprint_launch.txt").read_text(encoding="utf-8").splitlines():
    if line.startswith("=="):
        n_seg += 1
    elif "sha256:" in line:
        path, digest = line.split(" sha256:")
        launch.setdefault(pathlib.Path(path).name, set()).add(digest)
print("启动指纹：%d 段运行，每段记录的文件与 sha256（路径只留文件名）：" % n_seg)
for name, digests in sorted(launch.items()):
    print("  %s %s" % (name, ", ".join("sha256:" + d[:HASH_PREFIX_LEN] for d in sorted(digests))))
for p in COMPONENTS + [CONFIG_PATH]:
    assert launch[p.name] == {hashlib.sha256(p.read_bytes()).hexdigest()}, p.name
print("各段运行时的组件与配置与当前仓库逐字节相同（决策记录在运行后追加了复核，sha256 以上一行为准）。")

modules/m5_modeling/configs/s5_m10_wallclock.yaml sha256:3cc7a74dc131
registry/decision_records/DR-M5-016.yaml sha256:9d6d8721e56b
modules/m5_modeling/components/sgb_wallclock.py sha256:21e929b00379
modules/m5_modeling/components/sgb_cost_screen.py sha256:cf6088aa95be
modules/m5_modeling/components/method_screen_sgb.py sha256:2e54c618305f
modules/m5_modeling/components/method_screen.py sha256:05e19e025d0b
modules/m5_modeling/components/g3_ssxgb.py sha256:0f55ee4a8826
决策记录: registry/decision_records/DR-M5-016.yaml | step: S5.M10 | 两进程并行时主动方明文直方图省多少墙钟（对照 S5.M7 单进程两方合计计算的节省）
seed: 11 | git: 67f1b65（工作区含本步未提交改动）
启动指纹：1 段运行，每段记录的文件与 sha256（路径只留文件名）：
  DR-M5-016.yaml sha256:680cc5366c7a
  g3_ssxgb.py sha256:0f55ee4a8826
  method_screen.py sha256:05e19e025d0b
  method_screen_sgb.py sha256:2e54c618305f
  s5_m10_wallclock.yaml sha256:3cc7a74dc131
  sgb_cost_screen.py sha256:cf6088aa95be
  sgb_wallclock.py sha256:21e929b00379
各段运行时的组件与配置与当前仓库逐字节相同（决策记录在运行后追加了复核，sha256 以上一行为准）。


## 1. 任务与规模

S5.M7 计时过的 UCI Bank、Santander G3、mock 重合 80，以及 Orange 大版流失（每方 300 列）。配置为 S5.M1 / S5.M5 选定的优化版联邦。

In [2]:
from modules.m5_modeling.components import sgb_wallclock as WC

P = WC.MS.Params(cfg)
T = {t["name"]: json.loads((M10 / ("timing__%s.json" % t["name"])).read_text(encoding="utf-8")) for t in m10["tasks"]}
size = pd.DataFrame([{"任务": n, "内地列": t["cols_guest"], "香港列": t["cols_host"], "训练行": t["rows"], "计时棵数": t["trees"],
                      "四次打分逐位相同": t["preds_identical"]} for n, t in T.items()])
print(size.to_string(index=False))

                任务  内地列  香港列    训练行  计时棵数  四次打分逐位相同
          uci_bank   12    7  32950    10      True
      santander_g3   19   25 135819    10      True
    mock_overlap80   20   25  62564    10      True
orange_large_churn  300  300  40000     5      True


## 2. 复现

下一格用仓库组件在 UCI Bank 上重新起两个进程，原版与明文各训前 2 棵，要求测试打分逐位相同。

In [3]:
NAME = m10["tasks"][0]["name"]
REPRO_TREES = WC.WARMUP_TREES
res = json.loads((RUN / (NAME + ".json")).read_text(encoding="utf-8"))
sc = WC.MS.Screen(NAME, cfg, PRIV, M10 / "unused")
REPRO_DIR = M10 / "repro"
REPRO_DIR.mkdir(exist_ok=True)
spec = {**WC.prepare(P, sc, res, m10["method"], REPRO_DIR), "base_config": cfg}
spec_path = REPRO_DIR / ("%s__spec.json" % NAME)
spec_path.write_text(json.dumps(spec, ensure_ascii=False), encoding="utf-8")
ports = [int(p) for p in m10["fed_ports"]]
runs = [WC.launch(spec_path, plain, REPRO_TREES, REPRO_DIR, "%s__repro_%s" % (NAME, WC.KINDS[int(plain)]), ports)
        for plain in (False, True)]
pathlib.Path(spec["data"]).unlink()
same = np.array_equal(*[np.load(REPRO_DIR / ("%s__pred.npy" % r["tag"])) for r in runs])
print("用仓库组件在 %s 上重跑两进程原版与明文（前 %d 棵）：打分逐位相同 %s；内地训练墙钟 %.1f / %.1f 秒" % (
    NAME, REPRO_TREES, same, runs[0]["fit_seconds"]["neidi"], runs[1]["fit_seconds"]["neidi"]))
assert same

用仓库组件在 uci_bank 上重跑两进程原版与明文（前 2 棵）：打分逐位相同 True；内地训练墙钟 3.7 / 2.6 秒


## 3. 结果

- **每棵树墙钟**：以内地进程的训练墙钟计。两方并行，两方的训练墙钟只差启动先后。
- **节省比例**：报告平均值，以及两对 ABBA 的范围；与 S5.M7 单进程两方合计计算的节省对照。
- **回环发送字节**：两个进程之间的全部通信，用来核对「被动方收到的消息不变」。
- 汇总 CSV 写入 `results/sgb_wallclock_v1/`。

In [4]:
BYTES_PER_MB = 1e6
OUT = REPO_ROOT / m10["results_dir"]
OUT.mkdir(parents=True, exist_ok=True)
R = m10["round_sec"]
s7 = pd.read_csv(REPO_ROOT / "modules/m5_modeling/results/sgb_cost_v1/timing.csv").set_index("task")["saving"].to_dict()
k7 = pd.read_csv(REPO_ROOT / "modules/m5_modeling/results/sgb_cost_v1/k_cost.csv")
LARGE_K = int(yaml.safe_load((REPO_ROOT / "modules/m5_modeling/configs/s5_m5_orange_large.yaml").read_text(encoding="utf-8"))["k_main"])
s7.update({k7["task"].iloc[0]: float(k7.loc[k7["k_per_party"] == LARGE_K, "saving"].iloc[0])})
wall = pd.DataFrame([{"task": n, "trees": t["trees"], "cols_guest": t["cols_guest"], "cols_host": t["cols_host"],
                      "wall_per_tree_base": t["wall_per_tree_base"], "wall_per_tree_plain": t["wall_per_tree_plain"],
                      "saving_wall": t["saving"], "saving_low": t["saving_range"][0], "saving_high": t["saving_range"][1],
                      "saving_s5m7_single_process": s7.get(n), "loopback_mb_base": t["loopback_tx_bytes_base"] / BYTES_PER_MB,
                      "loopback_mb_plain": t["loopback_tx_bytes_plain"] / BYTES_PER_MB, "preds_identical": t["preds_identical"]}
                     for n, t in T.items()])
runs = pd.DataFrame([{"task": n, "order": i + 1, "kind": "plain" if r["plain"] else "base", "trees": r["trees"],
                      "fit_seconds_neidi": r["fit_seconds"]["neidi"], "fit_seconds_hk": r["fit_seconds"]["hk"],
                      "launch_wall_seconds": r["launch_wall_seconds"], "loopback_tx_bytes": r["loopback_tx_bytes"]}
                     for n, t in T.items() for i, r in enumerate(t["runs"])])
wall.to_csv(OUT / "wallclock.csv", index=False, float_format="%.4f")
runs.to_csv(OUT / "wallclock_runs.csv", index=False, float_format="%.2f")
print("两进程并行的每棵树墙钟（内地训练墙钟；ABBA 两对范围），与 S5.M7 单进程两方合计计算的节省对照：")
print(wall.round(R + 1).to_string(index=False))
print("各次运行：")
print(runs.round(R).to_string(index=False))

两进程并行的每棵树墙钟（内地训练墙钟；ABBA 两对范围），与 S5.M7 单进程两方合计计算的节省对照：
              task  trees  cols_guest  cols_host  wall_per_tree_base  wall_per_tree_plain  saving_wall  saving_low  saving_high  saving_s5m7_single_process  loopback_mb_base  loopback_mb_plain  preds_identical
          uci_bank     10          12          7               2.976                1.788        0.399       0.376        0.421                       0.401            97.209             98.735             True
      santander_g3     10          19         25              22.358               15.342        0.314       0.305        0.323                       0.285           591.185            592.795             True
    mock_overlap80     10          20         25              22.017               14.750        0.330       0.321        0.339                       0.316           368.274            371.040             True
orange_large_churn      5         300        300              47.454               28.013        0.410    

## 4. 结论

1. **两方并行时，墙钟节省与单进程合计计算的节省基本相同**：

| 任务 | 每棵树墙钟（原版 → 明文） | 墙钟节省（两对范围） | S5.M7 单进程合计节省 |
|---|---|---|---|
| UCI Bank | 2.98 → 1.79 秒 | 40%（38% ~ 42%） | 40% |
| Santander G3 | 22.4 → 15.3 秒 | 31%（31% ~ 32%） | 29% |
| mock 重合 80 | 22.0 → 14.7 秒 | 33%（32% ~ 34%） | 32% |
| Orange 大版流失（每方 300 列） | 47.5 → 28.0 秒 | 41%（38% ~ 44%） | 43% |

2. **可能的原因**（没有逐段剖析，只是推测）：原版里主动方要对自己的特征做密文分桶求和，还要解密两方的直方图，这部分多数时间在关键路径上；省掉之后，墙钟按接近的比例缩短。
3. **被动方收到的消息不变**：两种版本的回环发送字节平均相差不到 2%，与同一版本两次之间的波动同量级（计数含容器内其他回环流量）；打分逐位相同。
4. **并行本身的收益**：两进程的原版墙钟比单进程合计时间短（如 Orange 大版流失 47.5 对 73.1 秒），说明两方仍有部分计算重叠。

**对报告的意义**：S5.M7 留下的开放问题「部署时墙钟能省多少」在本机两进程上得到实测：与单进程合计计算的节省基本相同，约三到四成。

**边界**：
- 容器 2 核、本机回环，没有跨境网络延迟；网络延迟占比高时，节省比例会被稀释；
- 只训前 10 棵（大版 5 棵）；
- 明文直方图只在实验代码里替换，用于正式联调需经安全评审；
- 公开数据上的开发证据，不换算成汇丰的时长或收益；技术侧自评，未经法务复核（W-001）。